# 🏷️ Week 6: Part-of-Speech (POS) Tagging Bahasa Indonesia

## Tujuan Pembelajaran:
1. Memahami konsep **POS Tagging** — proses pemberian label kelas kata pada setiap token (kata benda, kata kerja, kata sifat, dll.)
2. Menggunakan **spaCy** untuk POS tagging berbahasa Indonesia
3. Menerapkan deteksi bahasa (*language detection*) dengan **langdetect** sebelum melakukan tagging
4. Menganalisis distribusi POS tag dari seluruh dataset ulasan Honest Review
5. Menemukan kata (token) yang paling sering muncul berdasarkan kelas katanya

---
> 📂 **Dataset**: Honest Review (`cleandata.csv`) — kolom `text_final` (teks yang sudah dipreproses)
> File ini berada di folder `Week 3/` dalam workspace yang sama.
>
> ℹ️ **spaCy** digunakan sebagai pengganti Polyglot karena lebih kompatibel dengan Windows dan Python 3.x.


## 1) Install Library

Install **spaCy** dan **langdetect**. spaCy menyediakan model POS tagging berbahasa Indonesia (`id_core_news_sm`).


In [4]:
%pip install spacy langdetect


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Install model spaCy bahasa Indonesia dengan cara yang lebih stabil di Jupyter
%pip install -q "spacy>=3.7.0,<3.8.0"
%pip install -q https://github.com/explosion/spacy-models/releases/download/id_core_news_sm-3.7.0/id_core_news_sm-3.7.0-py3-none-any.whl

print("Model id_core_news_sm berhasil diinstall!")


CalledProcessError: Command '['c:\\Users\\mikba\\Downloads\\Documents\\PBA\\PBA\\.venv\\Scripts\\python.exe', '-m', 'spacy', 'download', 'id_core_news_sm']' returned non-zero exit status 1.

## 2) Import Library

Import modul yang dibutuhkan untuk POS tagging dan deteksi bahasa.

In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

import spacy
from langdetect import detect, LangDetectException

# Load model spaCy bahasa Indonesia
nlp = spacy.load("id_core_news_sm")

print("Library berhasil diimport!")
print(f"spaCy version: {spacy.__version__}")


## 3) Load Dataset

Load dataset Honest Review dari `cleandata.csv`. Kolom `text_final` berisi teks ulasan yang sudah melalui preprocessing (tokenisasi, stopword removal, stemming).


In [ ]:
file_path = '../Week 3/cleandata.csv'
df_text = pd.read_csv(file_path)

print(f"Jumlah data: {len(df_text)}")
print(f"Kolom: {list(df_text.columns)}")
df_text['text_final'].head()


## 4) Pra-Pemrosesan Teks untuk Polyglot

Polyglot sensitif terhadap karakter non-ASCII dan karakter kontrol. Kita perlu membersihkan teks terlebih dahulu sebelum melakukan tagging.

Fungsi `clean_text()` menghapus:
- Karakter *non-printable* (karakter kontrol)
- Karakter di luar rentang ASCII standard

In [ ]:
def clean_text(text):
    """
    Membersihkan teks dari karakter non-printable dan non-ASCII
    agar kompatibel dengan Polyglot.
    """
    text = str(text)
    # Hapus karakter non-printable (kategori 'C' = control characters)
    text = ''.join(char for char in text if unicodedata.category(char)[0] != 'C')
    # Hapus karakter non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

# Terapkan pembersihan ke kolom text_final
df_text['text_clean'] = df_text['text_final'].apply(clean_text)
print("Pembersihan teks selesai.")

# Tampilkan contoh
print("\nContoh teks setelah dibersihkan:")
print(df_text['text_clean'].iloc[0][:200])


## 5) Deteksi Bahasa dengan langdetect

Sebelum POS tagging, deteksi bahasa dari tiap teks untuk memastikan model yang tepat digunakan. Model POS spaCy bersifat bahasa-spesifik (`id_core_news_sm` untuk Bahasa Indonesia).


In [ ]:
def detect_language(text):
    """
    Mendeteksi bahasa teks menggunakan langdetect.
    Return kode bahasa (misal: 'id', 'en') atau 'unknown'.
    """
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'


print("Deteksi bahasa pada 10 data pertama:")
print(f"{'No.':<5} {'Kode':<8} {'Cuplikan Teks'}")
print('-' * 65)

for idx, row in df_text.iterrows():
    lang_code = detect_language(row['text_clean'])
    cuplikan = row['text_clean'][:40] + '...'
    print(f"{idx+1:<5} {lang_code:<8} {cuplikan}")
    if idx == 9:
        break


## 6) POS Tagging pada Satu Dokumen

Demonstrasi POS tagging pada dokumen pertama dalam dataset. Setiap token akan diberi label kelas kata menggunakan tag Universal POS (UPOS) dari spaCy.

| Tag  | Kelas Kata       |
|------|------------------|
| NOUN | Kata Benda       |
| VERB | Kata Kerja       |
| ADJ  | Kata Sifat       |
| ADV  | Kata Keterangan  |
| PRON | Kata Ganti       |
| DET  | Determiner       |
| ADP  | Kata Depan (Preposisi) |
| NUM  | Numeralia        |
| PUNCT| Tanda Baca       |
| PROPN| Kata Nama Diri   |


In [ ]:
# Ambil dokumen pertama
doc_sample = df_text['text_clean'].iloc[0]
print("Teks sample:")
print(doc_sample[:200])
print("\n" + '='*50)


In [ ]:
# POS Tagging dengan spaCy
doc = nlp(doc_sample)

print(f"{'Kata':<20} {'POS Tag':<10} {'Keterangan'}")
print('-' * 50)
for token in doc:
    print(f"{token.text:<20} {token.pos_:<10} {spacy.explain(token.pos_) or ''}")


## 7) POS Tagging pada Seluruh Dataset

Terapkan POS tagging ke seluruh teks dalam dataset. Proses ini dapat memakan waktu tergantung ukuran dataset.

In [ ]:
all_pos_tags = []  # List untuk menampung hasil POS tags semua dokumen

for idx, text in df_text['text_clean'].items():
    try:
        doc = nlp(text)
        pos_tags = [(token.text, token.pos_) for token in doc]
        all_pos_tags.append(pos_tags)
    except Exception as e:
        print(f"Error pada baris {idx}: {str(e)[:80]}")
        all_pos_tags.append([])

print(f"POS tagging selesai untuk {len(all_pos_tags)} dokumen.")

# Tampilkan contoh hasil 3 dokumen pertama
print("\nContoh hasil POS tags (3 dokumen pertama):")
for i, tags in enumerate(all_pos_tags[:3]):
    print(f"  Dokumen {i+1} (5 token pertama): {tags[:5]}")


## 8) Analisis Distribusi POS Tag

Setelah mendapatkan semua POS tags, hitung:
- Jumlah total token per kelas kata
- Jumlah token unik per kelas kata

Ini memberikan gambaran komposisi linguistik dari keseluruhan dataset.

In [ ]:
# Ratakan semua POS tags menjadi satu list
flat_pos_tags = [item for sublist in all_pos_tags for item in sublist]

# Buat DataFrame dari hasil POS tagging
df_word_pos = pd.DataFrame(flat_pos_tags, columns=['Word', 'POS Tag'])

# Hitung frekuensi setiap POS tag
tag_counts = df_word_pos['POS Tag'].value_counts()

# Jumlah token unik per POS tag
unique_tokens_per_tag = df_word_pos.groupby('POS Tag')['Word'].nunique()

# Gabungkan ke dalam summary DataFrame
summary_df = pd.DataFrame({
    'POS Tag'      : tag_counts.index,
    'Total Token'  : tag_counts.values,
    'Token Unik'   : unique_tokens_per_tag
}).reset_index(drop=True)

summary_df = summary_df.sort_values('Total Token', ascending=False).reset_index(drop=True)

print(f"Total token keseluruhan: {len(flat_pos_tags):,}")
print(f"Jumlah tag unik        : {len(summary_df)}")
print("\nDistribusi POS Tag:")
print(summary_df.to_string(index=False))

# Simpan ke CSV
summary_df.to_csv('pos_tags_summary.csv', index=False)
print("\nHasil disimpan ke 'pos_tags_summary.csv'")

## 9) Token Paling Sering per Kelas Kata

Tampilkan 15 kombinasi kata + POS tag yang paling sering muncul dalam seluruh dataset. Ini membantu memahami kata-kata kunci yang dominan dalam ulasan Honest Review.


In [ ]:
# Hitung frekuensi setiap pasangan (Word, POS Tag)
top_entities = (
    df_word_pos
    .groupby(['Word', 'POS Tag'])
    .size()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={0: 'Frekuensi'})
)

print("Top 15 Token berdasarkan Frekuensi:")
print(top_entities.head(15).to_string(index=False))

# Simpan ke CSV
top_entities.to_csv('pos_top_tokens.csv', index=False)
print("\nHasil disimpan ke 'pos_top_tokens.csv'")

## 10) Visualisasi Distribusi POS Tag

Tampilkan distribusi frekuensi POS tag dalam bentuk bar chart untuk mendapatkan gambaran yang lebih intuitif.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(summary_df['POS Tag'], summary_df['Total Token'], color='steelblue', alpha=0.85, edgecolor='white')
plt.title('Distribusi POS Tag — Ulasan Honest Review', fontsize=13)
plt.xlabel('POS Tag')
plt.ylabel('Total Token')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
